# NutriVision AI — Team API Client Demo

**Purpose:** Test the ready FastAPI backend, not train any model. Requires only the `requests` library and a reachable running backend.

The backend owner must separately configure `NUTRIVISION_MODEL_DIR` and start `uvicorn backend.nutrivision_api:app` from the `NutriVision-AI` directory. This client notebook contains **no** model weights, images, passwords, or private paths.

In [ ]:
%pip -q install requests

## 1. Configure the server address

Use `http://127.0.0.1:8000` only when this notebook and the API server run on the **same computer/runtime**. If your teammate runs the backend elsewhere, replace it with the URL they provide. Do **not** expose the development API publicly without authentication and safeguards.

In [ ]:
import os
from pathlib import Path
import requests

API_URL = os.environ.get("NUTRIVISION_API_URL", "http://127.0.0.1:8000").rstrip("/")
print("API URL:", API_URL)

## 2. Check the API status

In [ ]:
response = requests.get(f"{API_URL}/health", timeout=15)
response.raise_for_status()
print(response.json())

## 3. Predict a food photo

Change `IMAGE_PATH` to a local file you have permission to use. The server returns an estimated **total plate** calorie value; ingredients and names are unverified suggestions.

In [ ]:
IMAGE_PATH = Path("sample_food.jpg")  # Change to your local JPG/PNG image
assert IMAGE_PATH.is_file(), f"Image not found: {IMAGE_PATH.resolve()}"
with IMAGE_PATH.open("rb") as fp:
    response = requests.post(
        f"{API_URL}/predict",
        files={"image": (IMAGE_PATH.name, fp)},
        data={"analyze_ingredients": "true"},
        timeout=180,
    )
response.raise_for_status()
result = response.json()
for field in ("prediction_id", "suggested_food", "estimated_calories", "food101_top3", "open_vocab_suggestions", "possible_ingredients", "requires_confirmation"):
    print(field, ":", result.get(field))

## 4. Save a **confirmed** correction (optional)

Only run after the user has inspected the original photo and confirmed the food name/ingredients. Feedback is saved in SQLite and does **not** retrain models or correct the calorie estimate.

In [ ]:
# Replace these examples with the user's verified answer before running.
USER_CONFIRMED_FOOD = ""  # Example after verification: "Chicken Kabsa"
USER_CONFIRMED_INGREDIENTS = []  # Example: ["chicken", "rice"]
if USER_CONFIRMED_FOOD.strip():
    feedback = requests.post(
        f"{API_URL}/feedback",
        json={
            "prediction_id": result["prediction_id"],
            "corrected_food": USER_CONFIRMED_FOOD.strip(),
            "confirmed_ingredients": USER_CONFIRMED_INGREDIENTS,
        },
        timeout=30,
    )
    feedback.raise_for_status()
    print(feedback.json())
else:
    print("Skipped: enter a user-confirmed food name to save feedback.")

## Notes for teammates

- The backend runs the heavy models; an API client does not require a GPU.
- The Food-101 standalone DINOv2 metric (~95.03% Top-1) does not apply to all arbitrary meals. The full ensemble's 95.34% score is a separate research result.
- Nutrition5k test MAE of 64.99 kcal is not a guarantee for Saudi or mixed meals.
- Never upload confidential images or share public API access without proper privacy/security controls.